# 🛒 Customer Churn Intelligence & Retention Dashboard
## AICTE | IBM SkillsBuild Data Analytics with AI Internship — Capstone Project

---

### Problem Statement

Retail businesses accumulate large volumes of transaction data but struggle to translate that data into actionable retention strategies. This project addresses two core challenges:

1. **Target leakage in churn prediction** — Naïvely computed RFM features often include information from the prediction period, inflating model performance and producing unreliable real-world predictions.
2. **Budget allocation** — Even with a reliable churn model, marketing teams lack tools to dynamically balance outreach cost against expected retained revenue.

### Approach

| Stage | Technique |
|-------|-----------|
| Data Cleaning | Column renaming, null imputation, type coercion, casing normalisation |
| Feature Engineering | Leakage-safe RFM with strict temporal split |
| Modelling | Logistic Regression + Decision Tree (80/20 stratified split) |
| Evaluation | Accuracy, Precision, Recall, ROC-AUC, Confusion Matrix |
| Deployment | Interactive Streamlit dashboard (`app.py`) |

### Dataset
UK Online Retail dataset — 541 910 raw transactions, 4 372 unique customers, Dec 2010 – Dec 2011.

---
## 1. Import Libraries & Data Loading

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay,
)

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

CLEAN_PATH = os.path.join('data', 'clean_data.csv')
RAW_PATH   = os.path.join('data', 'supermarket_sales_raw.csv')
PROFIT_MARGIN = 0.22

print('Libraries loaded successfully.')

In [ ]:
# ── Load clean_data.csv (pre-processed by app.py pipeline) ────────────────
# If clean_data.csv does not exist, run the cleaning pipeline below (Section 2)
# before re-executing this cell.

if os.path.exists(CLEAN_PATH):
    df = pd.read_csv(CLEAN_PATH, parse_dates=['Order_Date'])
    print(f'Loaded clean_data.csv  →  {len(df):,} rows  |  {df.shape[1]} columns')
else:
    print('clean_data.csv not found — run Section 2 first.')

df.head()

In [ ]:
# Data types and null counts
summary = pd.DataFrame({
    'dtype':    df.dtypes,
    'nulls':    df.isnull().sum(),
    'null_%':   (df.isnull().mean() * 100).round(2),
    'unique':   df.nunique(),
    'sample':   df.iloc[0],
})
display(summary)

---
## 2. Exploratory Data Analysis & Cleaning Logic

This section documents the cleaning rules applied by `load_and_clean_data()` in `app.py` and reproduces key quality checks inline.

| Step | Rule |
|------|------|
| 1 | Load raw CSV (latin-1 encoding) or generate synthetic data |
| 2 | Rename UK Online-Retail columns to canonical names |
| 3 | Drop rows with null `Customer_ID` (anonymous transactions) |
| 4 | Parse `Order_Date` with `format='mixed'`; drop unparseable rows |
| 5 | Remove returns / negative quantities (`Quantity <= 0`) |
| 6 | Strip currency symbols from monetary columns |
| 7 | Impute missing numeric values with column **median** |
| 8 | Impute missing categorical values with column **mode** |
| 9 | Title-case normalisation on `Product_Category` and `Region` |
| 10 | Derive `Revenue = Quantity × Unit_Price` and `Profit = Revenue × 22%` if absent |
| 11 | Clip `Revenue` and `Profit` to ≥ 0 |

In [ ]:
# ── Full cleaning pipeline (mirrors load_and_clean_data in app.py) ─────────
# Run this cell if you want to re-derive clean_data.csv from the raw file.

def _strip_currency(series):
    return (
        series.astype(str)
        .str.replace(r'[\u20b9$\xa3\u20acINR,\s]', '', regex=True)
        .replace('', np.nan)
        .astype(float)
    )

def _title_case_series(series):
    return series.astype(str).str.strip().str.title()

def run_cleaning_pipeline(file_path=RAW_PATH):
    raw = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
    print(f'Raw rows: {len(raw):,}  |  Raw nulls:\n{raw.isnull().sum()[raw.isnull().sum()>0]}')

    # Rename
    raw.rename(columns={
        'Invoice': 'Order_ID', 'StockCode': 'Product_Code',
        'Description': 'Product_Category', 'InvoiceDate': 'Order_Date',
        'Price': 'Unit_Price', 'Customer ID': 'Customer_ID', 'Country': 'Region',
    }, inplace=True)

    raw.dropna(subset=['Customer_ID'], inplace=True)
    raw['Customer_ID'] = raw['Customer_ID'].astype(str).str.strip()
    raw['Order_Date']  = pd.to_datetime(raw['Order_Date'], format='mixed', dayfirst=False, errors='coerce')
    raw.dropna(subset=['Order_Date'], inplace=True)
    raw = raw[raw['Quantity'] > 0].copy()

    for col in ['Unit_Price', 'Revenue', 'Profit']:
        if col in raw.columns:
            raw[col] = _strip_currency(raw[col])
    raw['Unit_Price'] = pd.to_numeric(raw['Unit_Price'], errors='coerce')

    for col in raw.select_dtypes(include=[np.number]).columns:
        if raw[col].isna().any():
            raw[col].fillna(raw[col].median(), inplace=True)
    for col in raw.select_dtypes(include=['object']).columns:
        if raw[col].isna().any():
            mode_val = raw[col].mode(dropna=True)
            if not mode_val.empty:
                raw[col].fillna(mode_val[0], inplace=True)

    for col in ['Product_Category', 'Region']:
        if col in raw.columns:
            raw[col] = _title_case_series(raw[col])

    if 'Revenue' not in raw.columns:
        raw['Revenue'] = np.round(raw['Quantity'] * raw['Unit_Price'], 2)
    if 'Profit' not in raw.columns:
        raw['Profit'] = np.round(raw['Revenue'] * PROFIT_MARGIN, 2)

    raw['Revenue'] = raw['Revenue'].clip(lower=0)
    raw['Profit']  = raw['Profit'].clip(lower=0)
    raw.reset_index(drop=True, inplace=True)

    os.makedirs('data', exist_ok=True)
    raw.to_csv(CLEAN_PATH, index=False)
    print(f'\nClean rows: {len(raw):,}  |  Saved to {CLEAN_PATH}')
    return raw

if os.path.exists(RAW_PATH):
    df = run_cleaning_pipeline()
else:
    print('Raw file not found — using already-loaded clean_data.csv.')

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────
print('=== Shape ===' )
print(df.shape)
print('\n=== Date Range ===')
print('Min:', df['Order_Date'].min().date(), '  Max:', df['Order_Date'].max().date())
print('\n=== Numeric Summary ===')
display(df[['Quantity','Unit_Price','Revenue','Profit']].describe())

In [ ]:
# ── Monthly Revenue Trend ─────────────────────────────────────────────────
monthly = (
    df.groupby(df['Order_Date'].dt.to_period('M'))['Revenue']
    .sum().reset_index()
)
monthly['Order_Date'] = monthly['Order_Date'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['Order_Date'], monthly['Revenue'], marker='o', linewidth=2, color='steelblue')
peak = monthly.loc[monthly['Revenue'].idxmax()]
low  = monthly.loc[monthly['Revenue'].idxmin()]
ax.annotate(f"Peak\n£{peak['Revenue']:,.0f}",
            xy=(peak['Order_Date'], peak['Revenue']),
            xytext=(10, -30), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=9)
ax.annotate(f"Low\n£{low['Revenue']:,.0f}",
            xy=(low['Order_Date'],  low['Revenue']),
            xytext=(10, 20),  textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red'),   color='red',   fontsize=9)
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 Product Categories by Revenue & Profit ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cat_rev = (
    df.groupby('Product_Category')['Revenue'].sum()
    .sort_values(ascending=False).head(10)
)
cat_rev.sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Categories by Revenue', fontweight='bold')
axes[0].set_xlabel('Revenue (£)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

cat_prof = (
    df.groupby('Product_Category')['Profit'].sum()
    .sort_values(ascending=False).head(10)
)
cat_prof.sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Top 10 Categories by Profit', fontweight='bold')
axes[1].set_xlabel('Profit (£)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Regional Revenue Distribution ─────────────────────────────────────────
reg = (
    df.groupby('Region')['Revenue'].sum()
    .sort_values(ascending=False)
)
top5   = reg.head(5)
others = pd.Series({'Other': reg.iloc[5:].sum()})
pie_data = pd.concat([top5, others])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Donut chart
wedges, texts, autotexts = axes[0].pie(
    pie_data.values, labels=pie_data.index,
    autopct=lambda p: f'{p:.1f}%' if p >= 2 else '',
    startangle=140, wedgeprops=dict(width=0.5),
    colors=sns.color_palette('muted', len(pie_data)),
)
axes[0].set_title('Customer Distribution by Region', fontweight='bold')

# Horizontal bar
reg.head(10).sort_values().plot.barh(ax=axes[1], color='teal')
axes[1].set_title('Top 10 Regions by Revenue', fontweight='bold')
axes[1].set_xlabel('Revenue (£)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

plt.tight_layout()
plt.show()

---
## 3. RFM Segmentation & Feature Engineering

### Leakage-Safe Temporal Split

```
Timeline ──────────────────────────────────────────────────────────►
            Historical Window              │  Observation Window
         (R, F, M computed here)           │  (Churn label defined here)
                                      cutoff_date
```

- **Recency (R)** — days since the customer's last purchase *before* the cutoff  
- **Frequency (F)** — number of unique invoices *before* the cutoff  
- **Monetary (M)** — total revenue *before* the cutoff  
- **Churn_Status** — `1` if the customer placed **zero** orders *on or after* the cutoff, else `0`

This guarantees no information from the future observation window contaminates the features.

In [ ]:
# ── Cutoff date (75% through the dataset timeline) ────────────────────────
min_date = df['Order_Date'].min()
max_date = df['Order_Date'].max()
CUTOFF   = min_date + (max_date - min_date) * 0.75
print(f'Dataset range : {min_date.date()}  →  {max_date.date()}')
print(f'Cutoff date   : {CUTOFF.date()}')
print(f'Historical window  : {min_date.date()} to {(CUTOFF - pd.Timedelta(days=1)).date()}')
print(f'Observation window : {CUTOFF.date()} to {max_date.date()}')

In [ ]:
# ── Build leakage-safe RFM table ──────────────────────────────────────────
hist = df[df['Order_Date'] < CUTOFF].copy()
obs  = df[df['Order_Date'] >= CUTOFF].copy()

last_purchase = hist.groupby('Customer_ID')['Order_Date'].max()
recency   = (CUTOFF - last_purchase).dt.days
frequency = hist.groupby('Customer_ID')['Order_ID'].nunique()
monetary  = hist.groupby('Customer_ID')['Revenue'].sum()

rfm = pd.DataFrame({'Recency': recency, 'Frequency': frequency, 'Monetary': monetary}).reset_index()

active_in_obs = set(obs['Customer_ID'].unique())
rfm['Churn_Status'] = rfm['Customer_ID'].apply(lambda c: 0 if c in active_in_obs else 1)

rfm['Recency']   = rfm['Recency'].clip(lower=0)
rfm['Frequency'] = rfm['Frequency'].clip(lower=1)
rfm['Monetary']  = rfm['Monetary'].clip(lower=0)

print(f'RFM table shape: {rfm.shape}')
print(f"Churn distribution:\n{rfm['Churn_Status'].value_counts().rename({0:'Retained',1:'Churned'})}")
rfm.head()

In [ ]:
# ── RFM feature distributions by churn status ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = {0: 'steelblue', 1: 'tomato'}
labels = {0: 'Retained', 1: 'Churned'}

for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    for status in [0, 1]:
        subset = rfm[rfm['Churn_Status'] == status][col]
        ax.hist(subset, bins=40, alpha=0.6, color=colors[status], label=labels[status],
                density=True)
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('RFM Feature Distributions: Retained vs Churned', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
corr = rfm[['Recency', 'Frequency', 'Monetary', 'Churn_Status']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('RFM Feature Correlation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Churn Prediction Model Training & Evaluation

Two classifiers are trained on the leakage-safe RFM features:

| Model | Notes |
|-------|-------|
| **Logistic Regression** | Features standardised with `StandardScaler`; `max_iter=1000` |
| **Decision Tree** | `max_depth=5`; raw RFM features (no scaling required) |

**Train / Test split:** 80 % / 20 % (stratified on `Churn_Status`, `random_state=42`)  
**Risk Tier assignment:** Quantile-based (33rd / 67th percentile) to prevent all customers collapsing into one tier.

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────
FEATURES = ['Recency', 'Frequency', 'Monetary']
TARGET   = 'Churn_Status'

X = rfm[FEATURES].values
y = rfm[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean():.2%}  |  Test churn rate: {y_test.mean():.2%}')

In [ ]:
# ── Fit models ────────────────────────────────────────────────────────────
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

print('Models trained.')

In [ ]:
# ── Evaluation metrics ────────────────────────────────────────────────────
results = {}

for name, model, X_ev in [
    ('Logistic Regression', lr, X_test_s),
    ('Decision Tree',       dt, X_test),
]:
    y_pred = model.predict(X_ev)
    y_prob = model.predict_proba(X_ev)[:, 1]
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred),                  4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0),4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0),   4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob),                   4),
        'cm':        confusion_matrix(y_test, y_pred),
        'fpr':       roc_curve(y_test, y_prob)[0],
        'tpr':       roc_curve(y_test, y_prob)[1],
        'y_prob':    y_prob,
    }

metrics_df = pd.DataFrame({
    k: {m: results[k][m] for m in ['Accuracy','Precision','Recall','ROC-AUC']}
    for k in results
}).T
print('=== Model Performance Scorecard ===')
display(metrics_df)

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name in zip(axes, ['Logistic Regression', 'Decision Tree']):
    cm = results[name]['cm']
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['No Churn', 'Churn'],
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{name}', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── ROC-AUC Curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['steelblue', 'darkorange']

for (name, color) in zip(['Logistic Regression', 'Decision Tree'], colors):
    auc = results[name]['ROC-AUC']
    ax.plot(results[name]['fpr'], results[name]['tpr'],
            color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-AUC Curves', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Assign quantile-based risk tiers & build scored table ─────────────────
all_X_s = scaler.transform(X)

rfm_scored = rfm.copy()
rfm_scored['LR_Churn_Prob'] = np.round(lr.predict_proba(all_X_s)[:, 1], 4)
rfm_scored['DT_Churn_Prob'] = np.round(dt.predict_proba(X)[:, 1], 4)

for prefix in ('LR', 'DT'):
    col      = f'{prefix}_Churn_Prob'
    tier_col = f'{prefix}_Risk_Tier'
    lo       = rfm_scored[col].quantile(0.33)
    hi       = rfm_scored[col].quantile(0.67)
    rfm_scored[tier_col] = rfm_scored[col].apply(
        lambda p: 'High Risk' if p >= hi else ('Medium Risk' if p >= lo else 'Low Risk')
    )

print('LR Risk Tier distribution:')
print(rfm_scored['LR_Risk_Tier'].value_counts().to_string())
print('\nDT Risk Tier distribution:')
print(rfm_scored['DT_Risk_Tier'].value_counts().to_string())

# Top 10 highest-risk customers (LR)
display(
    rfm_scored[['Customer_ID','Recency','Frequency','Monetary',
                'LR_Churn_Prob','LR_Risk_Tier','Churn_Status']]
    .sort_values('LR_Churn_Prob', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

In [ ]:
# ── Risk tier bar chart ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tier_order  = ['High Risk', 'Medium Risk', 'Low Risk']
tier_colors = ['tomato', 'gold', 'mediumseagreen']

for ax, prefix, label in zip(axes, ('LR', 'DT'), ('Logistic Regression', 'Decision Tree')):
    counts = rfm_scored[f'{prefix}_Risk_Tier'].value_counts().reindex(tier_order, fill_value=0)
    ax.bar(counts.index, counts.values, color=tier_colors, edgecolor='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
    ax.set_title(f'Risk Tier Distribution\n{label}', fontweight='bold')
    ax.set_ylabel('Number of Customers')
    ax.set_ylim(0, counts.max() * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# ── Retention Campaign ROI Calculator ─────────────────────────────────────
THRESHOLD        = 0.50
MAX_CUSTOMERS    = 200
COST_PER_CONTACT = 5.0    # GBP
CUSTOMER_LTV     = 250.0  # GBP

prob_col = 'LR_Churn_Prob'
targeted = (
    rfm_scored[rfm_scored[prob_col] >= THRESHOLD]
    .sort_values(prob_col, ascending=False)
    .head(MAX_CUSTOMERS)
)

n                = len(targeted)
campaign_cost    = round(n * COST_PER_CONTACT, 2)
avg_prob         = targeted[prob_col].mean() if n > 0 else 0.0
retained_revenue = round(n * CUSTOMER_LTV * avg_prob, 2)
roi_pct          = round((retained_revenue - campaign_cost) / campaign_cost * 100, 2) if campaign_cost > 0 else 0.0

print(f'Threshold        : {THRESHOLD}')
print(f'Targeted customers: {n}')
print(f'Campaign cost    : £{campaign_cost:,.2f}')
print(f'Est. retained rev: £{retained_revenue:,.2f}')
print(f'ROI              : {roi_pct:.1f}%')

---
## 5. Streamlit Dashboard — Launch Instructions

The full interactive dashboard is implemented in `app.py` and includes all five tabs explored in this notebook:

| Tab | Content |
|-----|---------|
| 📊 Executive Overview | Monthly trend, top categories, regional breakdown |
| 🔮 Churn Scoring | Colour-coded risk table, CSV export |
| 🧪 Model Diagnostics | Confusion matrix, ROC-AUC, precision-recall trade-off, campaign ROI |
| 💡 Business Insights | Observations, insights, hypotheses, recommendations |
| 📄 Export Report | Auto-generated `Project_Report.docx` download |

### How to Launch

Run the cell below to start the Streamlit server directly from this notebook. The dashboard will open at **http://localhost:8501**.

In [ ]:
# ── Launch the Streamlit dashboard ────────────────────────────────────────
# This opens the app in a new browser tab.
# Press the Stop (■) button in the notebook toolbar to shut it down.

import subprocess, sys, time, webbrowser

proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.headless', 'true',
     '--server.port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

time.sleep(3)   # give Streamlit a moment to start
webbrowser.open('http://localhost:8501')
print('Streamlit dashboard launched at http://localhost:8501')
print('Run proc.terminate() to stop the server.')

In [ ]:
# ── Stop the Streamlit server ─────────────────────────────────────────────
# Uncomment and run this cell when you are done.

# proc.terminate()
# print('Streamlit server stopped.')